# Protótipo Autônomo — Benchmark OAB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1/blob/main/diego_bispo.ipynb)

Este notebook é **autônomo**: ele usa apenas o arquivo `curadorias.csv` na raiz do repositório e não importa nada de `diego_bispo/`.

**Fluxo**
1. Setup do ambiente
2. Configuração e utilitários
3. Preparo dos dados
4. Download dos modelos
5. Respostas dos candidatos
6. Avaliação quantitativa das objetivas
7. Avaliação qualitativa reference das discursivas
8. Consolidado executivo


## 1. Setup do ambiente


In [ ]:
import os

REPO_URL = 'https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1'
WORKDIR = '/content/oab_prototipo'

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}

%cd {WORKDIR}
print('Diretório de trabalho:', os.getcwd())
print('curadorias.csv disponível?', os.path.exists('curadorias.csv'))


In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas numpy tqdm seaborn matplotlib huggingface_hub sentencepiece


## 2. Configuração e utilitários


In [ ]:
import gc
import json
import os
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from huggingface_hub import snapshot_download
from IPython.display import Image, Markdown, display
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from google.colab import userdata
except ImportError:
    userdata = None

ROOT = Path.cwd()
CURADORIAS_CSV = ROOT / 'curadorias.csv'
SAIDA_DIR = ROOT / 'saida_prototipo'
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = '/content/hf_cache'
os.environ['HF_HOME'] = HF_CACHE_DIR

MODELOS_CANDIDATOS = {
    'gemma2': 'google/gemma-2-2b-it',
    'llama323b': 'meta-llama/Llama-3.2-3B-Instruct',
    'llama321b': 'meta-llama/Llama-3.2-1B-Instruct',
}
MODELO_SBERT = 'rufimelo/Legal-BERTimbau-sts-large-ma-v3'

PESO_ARGUMENTACAO = 0.35
PESO_PRECISAO = 0.40
PESO_COESAO_LEGAL = 0.25
MAX_LENGTH_SBERT = 256

REPROCESSAR_RESPOSTAS = True
LIMITE_OBJ = None
LIMITE_DISC = None

QUESTOES_OBJ_CSV = SAIDA_DIR / 'questoes_objetivas_preparadas.csv'
QUESTOES_DISC_CSV = SAIDA_DIR / 'questoes_discursivas_preparadas.csv'
RESPOSTAS_OBJ_CSV = SAIDA_DIR / 'respostas_objetivas.csv'
RESPOSTAS_DISC_CSV = SAIDA_DIR / 'respostas_discursivas.csv'
BENCHMARK_OBJ_CSV = SAIDA_DIR / 'benchmark_objetivas.csv'
AVALIACAO_DISC_CSV = SAIDA_DIR / 'avaliacao_discursivas_reference.csv'
BENCHMARK_DISC_CSV = SAIDA_DIR / 'benchmark_discursivas_reference.csv'
CONSOLIDADO_CSV = SAIDA_DIR / 'benchmark_consolidado.csv'
VENCEDORES_DIFICULDADE_CSV = SAIDA_DIR / 'vencedores_por_dificuldade.csv'
VENCEDORES_DISCIPLINA_CSV = SAIDA_DIR / 'vencedores_por_disciplina.csv'
HEATMAP_CONSOLIDADO_PNG = SAIDA_DIR / 'heatmap_consolidado.png'

np.random.seed(42)
torch.manual_seed(42)

hf_token = os.environ.get('HF_TOKEN')
if userdata is not None and not hf_token:
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None
if not hf_token:
    raise ValueError('HF_TOKEN não encontrado. Configure nas Secrets do Colab ou no ambiente.')
os.environ['HF_TOKEN'] = hf_token

if not torch.cuda.is_available():
    raise RuntimeError('GPU não disponível. Ative uma GPU no ambiente de execução.')

print('GPU:', torch.cuda.get_device_name(0))
print('Saídas em:', SAIDA_DIR)


In [ ]:
PROMPT_CANDIDATO_DISCURSIVA = '''
Questão:
{questao}

Instruções complementares:
- Responda em português brasileiro, com linguagem jurídica formal e objetiva.
- Não invente base legal.
- Quando não souber o número exato do artigo, mencione apenas a lei aplicável ou o entendimento jurídico pertinente.
- Limite maximo de 30 linhas.

Escreva apenas a resposta final.
'''

PROMPT_CANDIDATO_OBJETIVA = '''
Responda a seguinte questão objetiva da prova da OAB.

{questao}

A) {A}
B) {B}
C) {C}
D) {D}

ATENÇÃO: Não inclua nada além do JSON. Se você incluir alguma coisa além do JSON, você será penalizado.
Responda apenas em JSON válido:
{{
  "resposta_objetiva": "A|B|C|D"
}}
'''


def salvar_csv(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'CSV salvo: {path}')


def normalize_text(text: object) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def to_float(value: object, default: float = 0.0) -> float:
    texto = normalize_text(value).replace(',', '.')
    if not texto:
        return default
    try:
        return float(texto)
    except ValueError:
        return default


def json_loads_safe(value: object, default):
    if value is None:
        return default
    texto = str(value).strip()
    if not texto:
        return default
    try:
        return json.loads(texto)
    except Exception:
        return default


def timestamp_execucao() -> str:
    return datetime.utcnow().isoformat(timespec='seconds') + 'Z'


def extrair_json_bruto(texto: str) -> str | None:
    if not texto:
        return None
    blocos_md = re.findall(r'```json\s*(\{.*?\})\s*```', texto, flags=re.S | re.I)
    for candidato in reversed(blocos_md):
        try:
            json.loads(candidato)
            return candidato
        except Exception:
            pass
    decoder = json.JSONDecoder()
    candidatos_validos = []
    for i, ch in enumerate(texto):
        if ch != '{':
            continue
        try:
            obj, consumed = decoder.raw_decode(texto[i:])
            candidatos_validos.append(texto[i:i + consumed])
        except Exception:
            continue
    return candidatos_validos[-1] if candidatos_validos else None

def montar_texto_discursivo(row: pd.Series) -> str:
    enunciado = normalize_text(row['Questão'])
    perguntas = json_loads_safe(row.get('Perguntas (J1)', '[]'), [])
    if not isinstance(perguntas, list) or not perguntas:
        return enunciado

    blocos = []
    for idx, item in enumerate(perguntas, start=1):
        if isinstance(item, dict):
            texto = normalize_text(item.get('texto'))
        else:
            texto = normalize_text(item)
        if texto:
            blocos.append(f'{idx}. {texto}')

    if not blocos:
        return enunciado
    return f"{enunciado}\n\nPerguntas:\n" + '\n\n'.join(blocos)


def preparar_dados(curadorias_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(curadorias_path, sep=';', dtype=str, keep_default_na=False)
    df['Pontuação Total (J1)'] = df['Pontuação Total (J1)'].apply(lambda x: to_float(x, 0.0))

    df_obj = df[df['Tipo Questão'] == 'OBJETIVA'].copy()
    df_disc = df[df['Tipo Questão'] != 'OBJETIVA'].copy()

    df_obj['alternativas'] = df_obj['Alternativas (J2)'].apply(lambda x: json_loads_safe(x, {}))
    df_obj['gabarito_oficial'] = df_obj['Gabarito'].astype(str).str.strip().str.upper()
    df_obj['texto_questao'] = df_obj['Questão'].astype(str).str.strip()
    df_obj['peso_questao'] = 1.0
    df_obj['nivel_dificuldade'] = df_obj['Dificuldade Nível'].replace('', 'nao_informado')
    df_obj['disciplina'] = df_obj['Especialidade Disciplina'].replace('', 'nao_informado')
    df_obj['tema'] = df_obj['Especialidade Tema'].replace('', 'nao_informado')

    df_disc['texto_questao'] = df_disc.apply(montar_texto_discursivo, axis=1)
    df_disc['gabarito_struct'] = df_disc['Gabarito'].apply(lambda x: json_loads_safe(x, {}))
    df_disc['gabarito_narrativo'] = df_disc['gabarito_struct'].apply(
        lambda d: normalize_text(d.get('gabarito_completo', '')) if isinstance(d, dict) else ''
    )
    df_disc['gabarito_itens'] = df_disc['gabarito_struct'].apply(
        lambda d: d.get('itens', []) if isinstance(d, dict) else []
    )
    df_disc['pontuacao_total'] = df_disc['Pontuação Total (J1)'].apply(lambda x: to_float(x, 0.0))
    df_disc['nivel_dificuldade'] = df_disc['Dificuldade Nível'].replace('', 'nao_informado')
    df_disc['disciplina'] = df_disc['Especialidade Disciplina'].replace('', 'nao_informado')
    df_disc['tema'] = df_disc['Especialidade Tema'].replace('', 'nao_informado')

    if LIMITE_OBJ:
        df_obj = df_obj.head(LIMITE_OBJ).copy()
    if LIMITE_DISC:
        df_disc = df_disc.head(LIMITE_DISC).copy()

    salvar_csv(
        df_obj[[
            'ID da Questão', 'Tipo Questão', 'Dataset', 'Prompt System', 'texto_questao', 'alternativas',
            'gabarito_oficial', 'nivel_dificuldade', 'disciplina', 'tema', 'peso_questao',
            'Número Questão Sequencial', 'Número Questão Exame', 'ID Exame', 'Ano Exame'
        ]].assign(alternativas=lambda d: d['alternativas'].apply(json.dumps)),
        QUESTOES_OBJ_CSV,
    )

    salvar_csv(
        df_disc[[
            'ID da Questão', 'Tipo Questão', 'Dataset', 'Prompt System', 'texto_questao', 'gabarito_narrativo',
            'gabarito_itens', 'pontuacao_total', 'nivel_dificuldade', 'disciplina', 'tema',
            'Número Questão Sequencial', 'Número Questão Exame', 'ID Exame', 'Ano Exame'
        ]].assign(gabarito_itens=lambda d: d['gabarito_itens'].apply(json.dumps)),
        QUESTOES_DISC_CSV,
    )

    return df, df_obj.reset_index(drop=True), df_disc.reset_index(drop=True)


def load_model(model_name: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=HF_CACHE_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=HF_CACHE_DIR,
        device_map='auto',
        quantization_config=bnb_config,
        low_cpu_mem_usage=True,
    )
    return model, tokenizer


def unload(model, tokenizer):
    try:
        model.to('cpu')
    except Exception:
        pass
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()


def gerar_texto(model, tokenizer, prompt: str, sample: bool, max_tokens: int, temperature: float = 0.7, system_prompt: str | None = None) -> str:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    messages_com_system = []
    if system_prompt and normalize_text(system_prompt):
        messages_com_system.append({'role': 'system', 'content': normalize_text(system_prompt)})
    messages_com_system.append({'role': 'user', 'content': prompt.strip()})

    messages_sem_system = [{
        'role': 'user',
        'content': f"{normalize_text(system_prompt)}\n\n{prompt.strip()}" if system_prompt else prompt.strip(),
    }]

    input_ids = None
    attention_mask = None

    for messages in (messages_com_system, messages_sem_system):
        try:
            result = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True)
            if hasattr(result, 'input_ids'):
                input_ids = result.input_ids.to(device)
                attention_mask = result.attention_mask.to(device) if hasattr(result, 'attention_mask') else torch.ones_like(input_ids)
            else:
                input_ids = result.to(device)
                attention_mask = torch.ones_like(input_ids)
            break
        except Exception:
            continue

    if input_ids is None:
        texto = f"{normalize_text(system_prompt)}\n\n{prompt.strip()}" if system_prompt else prompt.strip()
        encoded = tokenizer(texto, return_tensors='pt').to(device)
        input_ids = encoded.input_ids
        attention_mask = encoded.attention_mask

    generation_kwargs = {
        'max_new_tokens': max_tokens,
        'do_sample': sample,
        'pad_token_id': tokenizer.eos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'repetition_penalty': 1.3 if sample else 1.1,
        'attention_mask': attention_mask,
    }
    if sample:
        generation_kwargs['temperature'] = temperature
        generation_kwargs['top_p'] = 0.9

    with torch.no_grad():
        output = model.generate(input_ids=input_ids, **generation_kwargs)
    novo_texto = output[0][input_ids.shape[-1]:]
    return tokenizer.decode(novo_texto, skip_special_tokens=True).strip()


def gerar_resposta_objetiva(model, tokenizer, row: pd.Series, nome_modelo: str) -> dict:
    alternativas = row['alternativas'] if isinstance(row['alternativas'], dict) else json_loads_safe(row['alternativas'], {})
    prompt = PROMPT_CANDIDATO_OBJETIVA.format(
        questao=row['texto_questao'],
        A=alternativas.get('A', ''),
        B=alternativas.get('B', ''),
        C=alternativas.get('C', ''),
        D=alternativas.get('D', ''),
    )
    prefill = '{'
    saida = prefill + gerar_texto(
        model,
        tokenizer,
        prompt + f'\n{prefill}',
        sample=False,
        max_tokens=40,
        system_prompt=row.get('Prompt System'),
    )
    json_bruto = extrair_json_bruto(saida)
    try:
        resposta_json = json.loads(json_bruto) if json_bruto else {}
    except Exception:
        resposta_json = {}

    letra = normalize_text(resposta_json.get('resposta_objetiva', '')).upper()
    if letra not in {'A', 'B', 'C', 'D'}:
        letra = 'N/A'

    return {
        'question_id': row['ID da Questão'],
        'tipo_questao': row['Tipo Questão'],
        'dataset': row['Dataset'],
        'modelo': nome_modelo,
        'nivel_dificuldade': row['nivel_dificuldade'],
        'disciplina': row['disciplina'],
        'tema': row['tema'],
        'texto_questao': row['texto_questao'],
        'gabarito_oficial': row['gabarito_oficial'],
        'resposta': letra,
        'correto': letra == row['gabarito_oficial'],
        'json_parse_ok': bool(resposta_json),
        'saida_bruta': saida,
        'timestamp_execucao': timestamp_execucao(),
    }


def limpar_resposta_discursiva(texto: str) -> str:
    resposta = normalize_text(texto)
    resposta = re.sub(r'^resposta\s*:\s*', '', resposta, flags=re.I)
    resposta = re.sub(r'^resposta\s+final\s*:\s*', '', resposta, flags=re.I)
    return resposta.strip()


def gerar_resposta_discursiva(model, tokenizer, row: pd.Series, nome_modelo: str) -> dict:
    prompt = PROMPT_CANDIDATO_DISCURSIVA.format(questao=row['texto_questao'])
    resposta = gerar_texto(
        model,
        tokenizer,
        prompt,
        sample=True,
        max_tokens=700,
        temperature=0.7,
        system_prompt=row.get('Prompt System'),
    )
    resposta = limpar_resposta_discursiva(resposta)

    return {
        'question_id': row['ID da Questão'],
        'tipo_questao': row['Tipo Questão'],
        'dataset': row['Dataset'],
        'modelo': nome_modelo,
        'nivel_dificuldade': row['nivel_dificuldade'],
        'disciplina': row['disciplina'],
        'tema': row['tema'],
        'pontuacao_total': row['pontuacao_total'],
        'texto_questao': row['texto_questao'],
        'gabarito_narrativo': row['gabarito_narrativo'],
        'gabarito_itens': json.dumps(row['gabarito_itens'], ensure_ascii=False),
        'resposta': resposta,
        'timestamp_execucao': timestamp_execucao(),
    }


def gerar_respostas(df_obj: pd.DataFrame, df_disc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    respostas_obj = []
    respostas_disc = []

    for nome_modelo, model_name in MODELOS_CANDIDATOS.items():
        print(f'Processando candidato: {nome_modelo} -> {model_name}')
        model, tokenizer = load_model(model_name)

        print('  → Objetivas')
        for _, row in tqdm(df_obj.iterrows(), total=len(df_obj), desc=f'Objetivas | {nome_modelo}'):
            respostas_obj.append(gerar_resposta_objetiva(model, tokenizer, row, nome_modelo))

        print('  → Discursivas')
        for _, row in tqdm(df_disc.iterrows(), total=len(df_disc), desc=f'Discursivas | {nome_modelo}'):
            respostas_disc.append(gerar_resposta_discursiva(model, tokenizer, row, nome_modelo))

        unload(model, tokenizer)

    df_resp_obj = pd.DataFrame(respostas_obj)
    df_resp_disc = pd.DataFrame(respostas_disc)
    salvar_csv(df_resp_obj, RESPOSTAS_OBJ_CSV)
    salvar_csv(df_resp_disc, RESPOSTAS_DISC_CSV)
    return df_resp_obj, df_resp_disc


def avaliar_objetivas(df_resp_obj: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    detalhe = df_resp_obj.copy()
    detalhe['correto'] = detalhe['correto'].astype(bool)
    resumo = (
        detalhe.groupby('modelo', dropna=False)
        .agg(
            acuracia_objetivas=('correto', 'mean'),
            acertos_objetivas=('correto', 'sum'),
            total_objetivas=('correto', 'size'),
        )
        .reset_index()
        .sort_values(['acuracia_objetivas', 'acertos_objetivas', 'modelo'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    salvar_csv(resumo, BENCHMARK_OBJ_CSV)
    return detalhe, resumo


@dataclass
class GabaritoItem:
    texto: str
    peso_maximo: float | None
    secao: str | None = None


class EmbeddingBackend:
    def __init__(self, model_name: str, prefix: str = 'passage', batch_size: int = 8, max_length: int = MAX_LENGTH_SBERT):
        self.model_name = model_name
        self.prefix = prefix
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'[SBERT] Carregando {self.model_name}')
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name).to(self.device)
        self.model.eval()
        self.cache = {}

    def _format(self, text: str) -> str:
        text = normalize_text(text)
        return f'{self.prefix}: {text}' if self.prefix and text else text

    def _mean_pool(self, hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).expand(hidden_state.size()).float()
        soma = torch.sum(hidden_state * mask, dim=1)
        contagem = torch.clamp(mask.sum(dim=1), min=1e-9)
        return soma / contagem

    def get_embeddings(self, texts: Iterable[str]) -> dict[str, np.ndarray]:
        textos = [normalize_text(text) for text in texts]
        textos = [texto for texto in dict.fromkeys(textos) if texto]
        faltantes = [texto for texto in textos if texto not in self.cache]

        for start in range(0, len(faltantes), self.batch_size):
            lote = faltantes[start:start + self.batch_size]
            lote_modelo = [self._format(texto) for texto in lote]
            inputs = self.tokenizer(
                lote_modelo,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = self._mean_pool(outputs.last_hidden_state, inputs['attention_mask'])
                embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            for texto, emb in zip(lote, embeddings):
                self.cache[texto] = emb.detach().cpu().numpy()

        return {texto: self.cache[texto] for texto in textos if texto in self.cache}


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))


def carregar_itens_gabarito(gabarito_itens_json: str | list | None) -> list[GabaritoItem]:
    if isinstance(gabarito_itens_json, str):
        dados = json_loads_safe(gabarito_itens_json, [])
    elif isinstance(gabarito_itens_json, list):
        dados = gabarito_itens_json
    else:
        dados = []
    itens = []
    for item in dados:
        if isinstance(item, dict) and normalize_text(item.get('texto')):
            itens.append(GabaritoItem(
                texto=normalize_text(item.get('texto')),
                peso_maximo=item.get('peso_maximo'),
                secao=item.get('secao'),
            ))
    return itens


def extrair_secoes_ordenadas(itens: list[GabaritoItem]) -> list[str]:
    vistas = set()
    secoes = []
    for item in itens:
        secao = normalize_text(item.secao)
        if secao and secao not in vistas:
            vistas.add(secao)
            secoes.append(secao)
    return secoes


def score_precisao(resposta: str, itens: list[GabaritoItem], backend: EmbeddingBackend) -> float:
    resposta = normalize_text(resposta)
    if not resposta:
        return 0.0
    itens_com_peso = [item for item in itens if item.peso_maximo is not None]
    if not itens_com_peso:
        return 0.0

    textos_item = [normalize_text(item.texto) for item in itens_com_peso if normalize_text(item.texto)]
    if not textos_item:
        return 0.0

    embeddings = backend.get_embeddings([resposta] + textos_item)
    if resposta not in embeddings:
        return 0.0

    soma_ponderada = 0.0
    soma_pesos = 0.0
    for item, texto_item in zip(itens_com_peso, textos_item):
        if texto_item not in embeddings:
            continue
        sim = cosine_sim(embeddings[resposta], embeddings[texto_item])
        peso = to_float(item.peso_maximo, 0.0)
        soma_ponderada += sim * peso
        soma_pesos += peso
    return (soma_ponderada / soma_pesos) if soma_pesos else 0.0


def score_argumentacao(resposta: str, secoes: list[str], backend: EmbeddingBackend) -> float:
    resposta = normalize_text(resposta)
    secoes = [normalize_text(secao) for secao in secoes if normalize_text(secao)]
    if not resposta or not secoes:
        return 0.0
    embeddings = backend.get_embeddings([resposta] + secoes)
    if resposta not in embeddings:
        return 0.0
    scores = [cosine_sim(embeddings[resposta], embeddings[secao]) for secao in secoes if secao in embeddings]
    return float(np.mean(scores)) if scores else 0.0


def score_coesao_legal(resposta: str, gabarito_narrativo: str, backend: EmbeddingBackend) -> float:
    resposta = normalize_text(resposta)
    gabarito_narrativo = normalize_text(gabarito_narrativo)
    if not resposta or not gabarito_narrativo:
        return 0.0
    embeddings = backend.get_embeddings([resposta, gabarito_narrativo])
    if resposta not in embeddings or gabarito_narrativo not in embeddings:
        return 0.0
    return cosine_sim(embeddings[resposta], embeddings[gabarito_narrativo])


def avaliar_discursivas_reference(df_resp_disc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    backend = EmbeddingBackend(MODELO_SBERT, prefix='passage', max_length=MAX_LENGTH_SBERT)
    linhas = []

    for _, row in tqdm(df_resp_disc.iterrows(), total=len(df_resp_disc), desc='Avaliando discursivas (reference)'):
        itens = carregar_itens_gabarito(row['gabarito_itens'])
        secoes = extrair_secoes_ordenadas(itens)
        argumentacao = score_argumentacao(row['resposta'], secoes, backend)
        precisao = score_precisao(row['resposta'], itens, backend)
        coesao_legal = score_coesao_legal(row['resposta'], row['gabarito_narrativo'], backend)
        score_final = (
            (PESO_ARGUMENTACAO * argumentacao)
            + (PESO_PRECISAO * precisao)
            + (PESO_COESAO_LEGAL * coesao_legal)
        )
        pontuacao_total = to_float(row['pontuacao_total'], 0.0)
        nota_estimada = score_final * pontuacao_total
        linhas.append({
            'question_id': row['question_id'],
            'modelo': row['modelo'],
            'tipo_questao': row['tipo_questao'],
            'nivel_dificuldade': row['nivel_dificuldade'],
            'disciplina': row['disciplina'],
            'tema': row['tema'],
            'pontuacao_total': pontuacao_total,
            'argumentacao': round(argumentacao, 4),
            'precisao': round(precisao, 4),
            'coesao_legal': round(coesao_legal, 4),
            'score_final': round(score_final, 4),
            'nota_estimada': round(nota_estimada, 4),
        })

    detalhe = pd.DataFrame(linhas)
    resumo = (
        detalhe.groupby('modelo', dropna=False)
        .agg(
            argumentacao_media=('argumentacao', 'mean'),
            precisao_media=('precisao', 'mean'),
            coesao_legal_media=('coesao_legal', 'mean'),
            score_final_medio=('score_final', 'mean'),
            nota_estimada_total=('nota_estimada', 'sum'),
            pontuacao_total_discursivas=('pontuacao_total', 'sum'),
            total_discursivas=('question_id', 'size'),
        )
        .reset_index()
    )
    resumo['aproveitamento_discursivas'] = resumo['nota_estimada_total'] / resumo['pontuacao_total_discursivas'].replace(0, np.nan)
    resumo = resumo.fillna(0.0).sort_values(
        ['aproveitamento_discursivas', 'nota_estimada_total', 'modelo'],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    salvar_csv(detalhe, AVALIACAO_DISC_CSV)
    salvar_csv(resumo, BENCHMARK_DISC_CSV)
    return detalhe, resumo


def vencedores_por_grupo(df: pd.DataFrame, grupo_col: str, score_col: str, tipo_avaliacao: str, criterio: str) -> pd.DataFrame:
    base = df.copy().sort_values([grupo_col, score_col, 'modelo'], ascending=[True, False, True])
    vencedores = base.drop_duplicates(subset=[grupo_col]).copy()
    vencedores.insert(0, 'tipo_avaliacao', tipo_avaliacao)
    vencedores.insert(1, 'criterio', criterio)
    return vencedores


## 3. Preparo dos dados


In [ ]:
df_curadoria, df_obj, df_disc = preparar_dados(CURADORIAS_CSV)

print('Linhas curadas:', len(df_curadoria))
print('Objetivas:', len(df_obj))
print('Discursivas:', len(df_disc))

display(df_obj[[
    'ID da Questão', 'texto_questao', 'gabarito_oficial', 'nivel_dificuldade', 'disciplina'
]].head())

display(df_disc[[
    'ID da Questão', 'Tipo Questão', 'texto_questao', 'pontuacao_total', 'nivel_dificuldade', 'disciplina'
]].head())


## 4. Download dos modelos


In [ ]:
print('Baixando modelos uma única vez...')
for model_name in list(MODELOS_CANDIDATOS.values()) + [MODELO_SBERT]:
    print('Baixando:', model_name)
    snapshot_download(repo_id=model_name, cache_dir=HF_CACHE_DIR)


## 5. Respostas dos candidatos


In [ ]:
if REPROCESSAR_RESPOSTAS or not (RESPOSTAS_OBJ_CSV.exists() and RESPOSTAS_DISC_CSV.exists()):
    df_resp_obj, df_resp_disc = gerar_respostas(df_obj, df_disc)
else:
    df_resp_obj = pd.read_csv(RESPOSTAS_OBJ_CSV)
    df_resp_disc = pd.read_csv(RESPOSTAS_DISC_CSV)
    print('Reaproveitando respostas já salvas.')

display(df_resp_obj.head())
display(df_resp_disc.head())


## 6. Avaliação quantitativa das objetivas


In [ ]:
df_obj_detalhe, benchmark_obj = avaliar_objetivas(df_resp_obj)

melhor_obj = benchmark_obj.iloc[0]
display(Markdown(
    f"**Melhor modelo nas objetivas:** `{melhor_obj['modelo']}` com acurácia de **{melhor_obj['acuracia_objetivas']:.2%}** "
    f"({int(melhor_obj['acertos_objetivas'])}/{int(melhor_obj['total_objetivas'])})"
))

display(benchmark_obj.round(4))


## 7. Avaliação qualitativa reference das discursivas

A avaliação discursiva usa **um único SBERT jurídico** nas três dimensões:
- **Precisão:** cobertura ponderada dos itens do gabarito.
- **Argumentação:** presença da estrutura jurídica esperada pelas seções do espelho.
- **Coesão legal:** alinhamento global entre a resposta e o raciocínio jurídico narrativo do gabarito.


In [ ]:
df_disc_detalhe, benchmark_disc = avaliar_discursivas_reference(df_resp_disc)

melhor_disc = benchmark_disc.iloc[0]
display(Markdown(
    f"**Melhor modelo nas discursivas:** `{melhor_disc['modelo']}` com aproveitamento de **{melhor_disc['aproveitamento_discursivas']:.2%}** "
    f"e **{melhor_disc['nota_estimada_total']:.2f}** pontos estimados em **{melhor_disc['pontuacao_total_discursivas']:.2f}** possíveis."
))

display(benchmark_disc.round(4))
display(df_disc_detalhe.head().round(4))


## 8. Consolidado executivo


In [ ]:
benchmark_consolidado = benchmark_obj.merge(benchmark_disc, on='modelo', how='outer').fillna(0.0)
benchmark_consolidado['score_balanceado'] = (
    benchmark_consolidado['acuracia_objetivas'] + benchmark_consolidado['aproveitamento_discursivas']
) / 2
benchmark_consolidado['score_total_ponderado'] = (
    benchmark_consolidado['acertos_objetivas'] + benchmark_consolidado['nota_estimada_total']
) / (
    benchmark_consolidado['total_objetivas'] + benchmark_consolidado['pontuacao_total_discursivas']
).replace(0, np.nan)
benchmark_consolidado = benchmark_consolidado.fillna(0.0)
benchmark_consolidado = benchmark_consolidado.sort_values(
    ['score_balanceado', 'score_total_ponderado', 'modelo'],
    ascending=[False, False, True],
).reset_index(drop=True)

salvar_csv(benchmark_consolidado, CONSOLIDADO_CSV)

obj_dificuldade = (
    df_obj_detalhe.groupby(['nivel_dificuldade', 'modelo'], dropna=False)
    .agg(score=('correto', 'mean'), acertos=('correto', 'sum'), total=('correto', 'size'))
    .reset_index()
)

disc_dificuldade = (
    df_disc_detalhe.groupby(['nivel_dificuldade', 'modelo'], dropna=False)
    .agg(score_num=('nota_estimada', 'sum'), score_den=('pontuacao_total', 'sum'))
    .reset_index()
)
disc_dificuldade['score'] = disc_dificuldade['score_num'] / disc_dificuldade['score_den'].replace(0, np.nan)
disc_dificuldade = disc_dificuldade.fillna(0.0)

obj_disciplina = (
    df_obj_detalhe.groupby(['disciplina', 'modelo'], dropna=False)
    .agg(score=('correto', 'mean'), acertos=('correto', 'sum'), total=('correto', 'size'))
    .reset_index()
)

disc_disciplina = (
    df_disc_detalhe.groupby(['disciplina', 'modelo'], dropna=False)
    .agg(score_num=('nota_estimada', 'sum'), score_den=('pontuacao_total', 'sum'))
    .reset_index()
)
disc_disciplina['score'] = disc_disciplina['score_num'] / disc_disciplina['score_den'].replace(0, np.nan)
disc_disciplina = disc_disciplina.fillna(0.0)

vencedores_dificuldade = pd.concat([
    vencedores_por_grupo(obj_dificuldade, 'nivel_dificuldade', 'score', 'objetiva', 'nivel_dificuldade'),
    vencedores_por_grupo(disc_dificuldade, 'nivel_dificuldade', 'score', 'discursiva', 'nivel_dificuldade'),
], ignore_index=True)

vencedores_disciplina = pd.concat([
    vencedores_por_grupo(obj_disciplina, 'disciplina', 'score', 'objetiva', 'disciplina'),
    vencedores_por_grupo(disc_disciplina, 'disciplina', 'score', 'discursiva', 'disciplina'),
], ignore_index=True)

salvar_csv(vencedores_dificuldade, VENCEDORES_DIFICULDADE_CSV)
salvar_csv(vencedores_disciplina, VENCEDORES_DISCIPLINA_CSV)

melhor_geral = benchmark_consolidado.iloc[0]

display(Markdown(f'''
### Painel executivo
- **Melhor nas objetivas:** `{melhor_obj['modelo']}` com **{melhor_obj['acuracia_objetivas']:.2%}** de acurácia.
- **Melhor nas discursivas:** `{melhor_disc['modelo']}` com **{melhor_disc['aproveitamento_discursivas']:.2%}** de aproveitamento e **{melhor_disc['nota_estimada_total']:.2f} pontos**.
- **Melhor no consolidado balanceado:** `{melhor_geral['modelo']}` com **{melhor_geral['score_balanceado']:.2%}**.
'''))

display(benchmark_consolidado.round(4))

display(Markdown('### Vencedores por nível de dificuldade'))
display(vencedores_dificuldade.round(4))

display(Markdown('### Vencedores por disciplina'))
display(vencedores_disciplina.round(4))

heatmap_cols = [
    'acuracia_objetivas',
    'aproveitamento_discursivas',
    'argumentacao_media',
    'precisao_media',
    'coesao_legal_media',
    'score_balanceado',
]
heatmap_df = benchmark_consolidado.set_index('modelo')[heatmap_cols]

plt.figure(figsize=(12, 4.8))
sns.heatmap(heatmap_df, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, cbar=True)
plt.title('Painel consolidado de desempenho por modelo')
plt.xlabel('Métrica')
plt.ylabel('Modelo')
plt.tight_layout()
plt.savefig(HEATMAP_CONSOLIDADO_PNG, dpi=200, bbox_inches='tight')
plt.show()

display(Image(filename=str(HEATMAP_CONSOLIDADO_PNG)))
